In [3]:
import numpy as np
import pandas as pd

# =============================================================================
# 1. STANDARDIZED KINEMATIC & ENVIRONMENTAL CONSTANTS
# =============================================================================
va = 15.0                  # Uniform Airspeed [m/s]
R_min = 45.0               # Minimum physical turn radius [m]
omega_max = va / R_min     # Hard saturation yaw rate constraint [rad/s] (0.3333)
dt = 0.02                  # Standardized calculation time-step [s]
T_max = 100.0              # Full observation window [s]
steps = int(T_max / dt)

def wrap(angle):
    """Wraps an angle within [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

# =============================================================================
# 2. PERFORMANCE RECORDER ENGINE
# =============================================================================
def calculate_settling_time(xte_array, threshold, time_step=dt):
    """Calculates time taken to permanently settle within an error bound."""
    above_indices = np.where(xte_array >= threshold)[0]
    if len(above_indices) == 0:
        return 0.0
    last_above = above_indices[-1]
    if last_above == len(xte_array) - 1:
        return None  # Did Not Converge (DNC)
    return (last_above + 1) * time_step

def evaluate_metrics(xte_array):
    """Generates precise spatial accuracy metrics."""
    mean_xte = np.mean(xte_array)
    max_xte = np.max(xte_array)
    rmse = np.sqrt(np.mean(xte_array**2))
    final_xte = xte_array[-1]
    ts_5m = calculate_settling_time(xte_array, 5.0)
    ts_2m = calculate_settling_time(xte_array, 2.0)

    return {
        "Mean XTE (m)": round(mean_xte, 2),
        "Max XTE (m)": round(max_xte, 2),
        "RMSE (m)": round(rmse, 2),
        "Final XTE (m)": round(final_xte, 4),
        "Settling 5m (s)": f"{ts_5m:.2f}" if ts_5m is not None else "DNC",
        "Settling 2m (s)": f"{ts_2m:.2f}" if ts_2m is not None else "DNC"
    }

# =============================================================================
# 3. CONVERGED ALGORITHM GENERATORS
# =============================================================================

# --- CARROT CHASING ---
def run_carrot_line(delta=30.0, kappa=1.5):
    x, y, psi = 100.0, 0.0, 0.0  # Normalized Straight Line State
    theta = np.pi / 4
    xte_list = []
    for _ in range(steps):
        d = -x * np.sin(theta) + y * np.cos(theta)
        R_proj = x * np.cos(theta) + y * np.sin(theta)
        xt = (R_proj + delta) * np.cos(theta)
        yt = (R_proj + delta) * np.sin(theta)
        psi_d = np.arctan2(yt - y, xt - x)
        u = np.clip(kappa * wrap(psi_d - psi), -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

def run_carrot_loiter(delta=30.0, kappa=1.5, r=100.0):
    x, y, psi = 150.0, 0.0, np.pi/2  # Normalized Loiter State
    xte_list = []
    for _ in range(steps):
        d_radial = np.hypot(x, y)
        d = d_radial - r
        theta = np.arctan2(y, x)
        theta_t = theta + (delta / r)
        xt = r * np.cos(theta_t)
        yt = r * np.sin(theta_t)
        psi_d = np.arctan2(yt - y, xt - x)
        u = np.clip(kappa * wrap(psi_d - psi), -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

# --- NLGL ---
def run_nlgl_line(L=50.0):
    x, y, psi = 100.0, 0.0, 0.0
    theta = np.pi / 4
    xte_list = []
    for _ in range(steps):
        d = -x * np.sin(theta) + y * np.cos(theta)
        R_proj = x * np.cos(theta) + y * np.sin(theta)
        s = R_proj + np.sqrt(max(0, L**2 - d**2)) if abs(d) < L else R_proj
        xt, yt = s * np.cos(theta), s * np.sin(theta)
        eta = wrap(np.arctan2(yt - y, xt - x) - psi)
        u = np.clip((2.0 * va * np.sin(eta)) / L, -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

def run_nlgl_loiter(L=50.0, r=100.0):
    x, y, psi = 150.0, 0.0, np.pi/2
    xte_list = []
    for _ in range(steps):
        d_radial = np.hypot(x, y)
        d = d_radial - r
        theta = np.arctan2(y, x)
        cos_val = np.clip((d_radial**2 + r**2 - L**2) / (2.0 * d_radial * r + 1e-9), -1.0, 1.0)
        xt = r * np.cos(theta + np.arccos(cos_val))
        yt = r * np.sin(theta + np.arccos(cos_val))
        eta = wrap(np.arctan2(yt - y, xt - x) - psi)
        u = np.clip((2.0 * va * np.sin(eta)) / L, -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

# --- VECTOR FIELD ---
def run_vf_line(k=0.02, chi_infty=np.pi/4):
    x, y, psi = 100.0, 0.0, 0.0
    theta = np.pi / 4
    xte_list = []
    for _ in range(steps):
        d = -x * np.sin(theta) + y * np.cos(theta)
        chi_d = theta - chi_infty * np.tanh(k * d)
        u = np.clip(2.0 * wrap(chi_d - psi), -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

def run_vf_loiter(k=0.05, r=100.0):
    x, y, psi = 150.0, 0.0, np.pi/2
    xte_list = []
    for _ in range(steps):
        d_radial = np.hypot(x, y)
        d = d_radial - r
        theta = np.arctan2(y, x)
        chi_d = theta + np.pi/2 + np.arctan(k * d)
        u = np.clip(2.0 * wrap(chi_d - psi), -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

# --- ADAPTIVE LQR ---
def run_lqr_line(q22=5.0, db=100.0):
    x, y, psi = 100.0, 0.0, 0.0
    theta = np.pi / 4
    xte_list = []
    for _ in range(steps):
        d = -x * np.sin(theta) + y * np.cos(theta)
        vd = va * np.sin(psi - theta)
        denom = db - d if abs(db - d) > 1e-6 else 1e-6 * np.sign(db - d)
        q11 = abs(db / denom)
        u = -(np.sqrt(q11) * d + np.sqrt(2.0 * np.sqrt(q11) + q22) * vd)
        u = np.clip(u, -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

def run_lqr_loiter(q22=5.0, db=100.0, r=100.0):
    x, y, psi = 150.0, 0.0, np.pi/2
    xte_list = []
    for _ in range(steps):
        d_radial = np.hypot(x, y)
        d = -(d_radial - r)
        theta = np.arctan2(y, x) + np.pi/2
        vd = va * np.sin(psi - theta)
        denom = db - d if abs(db - d) > 1e-6 else 1e-6 * np.sign(db - d)
        q11 = abs(db / denom)
        u = -(np.sqrt(q11) * d + np.sqrt(2.0 * np.sqrt(q11) + q22) * vd)
        u = np.clip(u, -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d_radial - r))
    return np.array(xte_list)

# --- PROPORTIONAL PURSUIT LINE-OF-SIGHT (PLOS) ---
def run_plos_line(k1=2.0, k2=0.02):
    x, y, psi = 100.0, 0.0, 0.0
    theta = np.pi / 4
    xte_list = []
    for _ in range(steps):
        d = -x * np.sin(theta) + y * np.cos(theta)
        psi_d = theta - np.arctan(k2 * d)
        u = np.clip(k1 * wrap(psi_d - psi), -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

def run_plos_loiter(k1=2.0, k2=0.02, r=100.0):
    x, y, psi = 150.0, 0.0, np.pi/2
    xte_list = []
    for _ in range(steps):
        d_radial = np.hypot(x, y)
        d = d_radial - r
        theta = np.arctan2(y, x)
        psi_d = theta + np.pi/2 - np.arctan(k2 * d)
        u = np.clip(k1 * wrap(psi_d - psi), -omega_max, omega_max)
        x += va * np.cos(psi) * dt
        y += va * np.sin(psi) * dt
        psi = wrap(psi + u * dt)
        xte_list.append(abs(d))
    return np.array(xte_list)

# =============================================================================
# 4. EXECUTION & ANALYSIS DISPATCHER
# =============================================================================
scenarios = [
    ("Carrot Chasing (Line)",   run_carrot_line()),
    ("Carrot Chasing (Loiter)", run_carrot_loiter()),
    ("NLGL (Straight Line)",    run_nlgl_line()),
    ("NLGL (Circular Loiter)",  run_nlgl_loiter()),
    ("Vector Field (Line)",     run_vf_line()),
    ("Vector Field (Loiter)",   run_vf_loiter()),
    ("Adaptive LQR (Line)",     run_lqr_line()),
    ("Adaptive LQR (Loiter)",   run_lqr_loiter()),
    ("PLOS Guidance (Line)",    run_plos_line()),
    ("PLOS Guidance (Loiter)",  run_plos_loiter()),
]

master_records = []
for label, trajectory in scenarios:
    perf = evaluate_metrics(trajectory)
    perf["Algorithm/Scenario"] = label
    master_records.append(perf)

df = pd.DataFrame(master_records)[["Algorithm/Scenario", "Mean XTE (m)", "Max XTE (m)", "RMSE (m)", "Final XTE (m)", "Settling 5m (s)", "Settling 2m (s)"]]
print("\n" + "="*110 + f"\n{'UNIFIED UAV PATH-FOLLOWING MASTER BENCHMARK':^110}\n" + "="*110)
print(df.to_markdown(index=False))
print("="*110)


                                 UNIFIED UAV PATH-FOLLOWING MASTER BENCHMARK                                  
| Algorithm/Scenario      |   Mean XTE (m) |   Max XTE (m) |   RMSE (m) |   Final XTE (m) | Settling 5m (s)   | Settling 2m (s)   |
|:------------------------|---------------:|--------------:|-----------:|----------------:|:------------------|:------------------|
| Carrot Chasing (Line)   |           5.93 |         84    |      19.92 |          0      | 11.02             | 11.98             |
| Carrot Chasing (Loiter) |           3.3  |         50    |       8.8  |          1.413  | 6.38              | 7.10              |
| NLGL (Straight Line)    |           6    |         84    |      19.81 |          0      | 10.28             | 17.52             |
| NLGL (Circular Loiter)  |           2.32 |         50    |       8.85 |          0.0726 | 6.90              | 13.00             |
| Vector Field (Line)     |           7.35 |         84    |      21.06 |          0      | 16.6